# Проверка ЬД

In [10]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://qdrant:6333")

res = client.scroll(
    collection_name="ti_chunks",
    limit=1,
    with_payload=True,
)

points, _ = res

print("ID:", points[0].id)
print("Payload keys:", points[0].payload.keys())
print("\nTEXT:\n")
print(points[0].payload.get("text", "NO TEXT FIELD"))

ID: 0
Payload keys: dict_keys(['type', 'chunk_title', 'i', 'model', 'key', 'row', 'chunk_id', 'text'])

TEXT:

Введение
Настоящая технологическая инструкция регламентирует
последовательность технологических операций приготовления никелевого
католита, правила ведения и управления процессом, содержит характеристику
сырья, материалов, оборудования и товарной продукции, методы контроля и
метрологическое обеспечение, а также требования охраны труда и
промышленной безопасности.
Инструкция разработана в соответствии с проектом ФМ.04450-П-ИОС7.1-С
«ЦЭН. ОЭН-2. Электроэкстракция никеля из растворов хлорного растворения
ПНТП на объем производства 145 тыс.т/год электролитного никеля»,
выполненным в 2015 году ООО «Институт Гипроникель» на основании
Технологического регламента, выполненного по договору № 001-134н/0438-49-
14 от 01.04.2014 г. «Актуализация технологического регламента на
корректировку проекта переработки никелевого порошка трубчатых печей по
технологии хлорного выщелачивания с увелич

# Основной код

In [2]:
import os, json, time
import requests
from typing import List, Dict, Any

In [ ]:
# Вариант A: из контейнера (одна сеть compose)
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
QDRANT_URL = os.getenv("QDRANT_URL", "http://qdrant:6333")

# Вариант B: с хоста (если не видишь сервисные имена)
# OLLAMA_URL = "http://localhost:11434"
# QDRANT_URL = "http://localhost:6333"

COLLECTION = os.getenv("QDRANT_COLLECTION", "ti_chunks")

EMBED_MODEL = os.getenv("EMBED_MODEL", "bge-m3:latest")   # или nomic-embed-text
LLM_MODEL   = os.getenv("LLM_MODEL",   "qwen2.5:7b-instruct")         # пример, подставь свой

# print("OLLAMA_URL:", OLLAMA_URL)
# print("QDRANT_URL:", QDRANT_URL)
# print("COLLECTION:", COLLECTION)
# print("EMBED_MODEL:", EMBED_MODEL)
# print("LLM_MODEL:", LLM_MODEL)

def check_ollama():
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=20)
    r.raise_for_status()
    return r.json()

def check_qdrant():
    r = requests.get(f"{QDRANT_URL}/collections", timeout=20)
    r.raise_for_status()
    return r.json()

# print("Ollama:", check_ollama())
# print("Qdrant:", check_qdrant())

def ollama_pull(model: str):
    # В контейнере лучше pull делать через CLI, но через API тоже можно прогреть запросом.
    # Тут просто проверим, есть ли модель в tags.
    tags = check_ollama().get("models", [])
    names = {m.get("name") for m in tags}
    if model in names:
        print("OK already:", model)
    else:
        print("Model not in tags (pull via CLI):", model)
        print("Run: docker exec -it ollama ollama pull", model)

ollama_pull(EMBED_MODEL)
ollama_pull(LLM_MODEL)

In [ ]:
def embed(text: str) -> List[float]:
    r = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": EMBED_MODEL, "prompt": text},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["embedding"]

In [ ]:
def qdrant_search(vector: List[float], limit: int = 8) -> List[Dict[str, Any]]:
    r = requests.post(
        f"{QDRANT_URL}/collections/{COLLECTION}/points/search",
        json={
            "vector": vector,
            "limit": limit,
            "with_payload": True,
            "with_vector": False,
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["result"]

hits = qdrant_search(embed("Что такое католит?"), limit=5)
len(hits), hits

In [ ]:
qdrant_search(embed("Сравни определение католита и хлора?"), limit=5)

In [ ]:
def build_context(hits: List[Dict[str, Any]], max_chunks: int = 6, max_chars: int = 5000):
    parts = []
    citations = []
    total = 0

    for h in hits[:max_chunks]:
        p = h.get("payload") or {}
        text = p.get("text") or ""
        source = p.get("source") or p.get("file") or "unknown"
        chunk_id = p.get("chunk_id") or h.get("id")

        block = f"[{source}#{chunk_id}]\n{text}".strip()
        if not block:
            continue

        if total + len(block) > max_chars:
            break
        parts.append(block)
        citations.append({"source": source, "chunk_id": chunk_id, "score": h.get("score")})
        total += len(block)

    return "\n\n---\n\n".join(parts), citations

context, citations = build_context(hits)
print("context chars:", len(context))
context, citations

In [ ]:
# r = requests.post(
#     f"{OLLAMA_URL}/api/chat",
#     json={
#         "model": "bambucha/saiga-llama3:8b-q4_K",
#         "messages": [{"role":"user","content":"Привет! Одним словом: ок?"}],
#         "stream": False,
#         "options": {"num_ctx": 1024}
#     },
#     timeout=180,
# )
# print(r.status_code, r.text[:200])

In [ ]:
LLM_MODEL = "bambucha/saiga-llama3:8b-q4_K"

def generate_answer(question: str, context: str) -> str:
   r = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": LLM_MODEL,
        "messages": [
            {
              "role": "system",
              "content": """Ты помощник по техдокам.
            Используй ТОЛЬКО предоставленный контекст.
            Если в контексте нет ответа — напиши:
            "В контексте нет информации."
            Не придумывай источники и не используй интернет.
            Формат ссылок: [source#chunk_id].
            """
            },
            {
                "role": "user",
                "content": f"""
                Вопрос:
                {question}
                
                Контекст:
                {context}
                    """
            }
            ],
            "stream": False
        },
        timeout=180,
    )

    print("STATUS:", r.status_code)
    print("RAW:", r.text[:500])

    r.raise_for_status()
    return r.json()["message"]["content"]

question = "Что такое хлор?"
answer = generate_answer(question, context)
print(answer)

# Всё в одном

In [ ]:
def rag_ask(question: str, top_k: int = 10):
    vec = embed(question)
    hits = qdrant_search(vec, limit=top_k)
    print(hits)
    context, citations = build_context(hits)
    answer = generate_answer(question, context)
    return {"answer": answer, "citations": citations, "hits": hits}

out = rag_ask( "Что такое хлор?", top_k=12)
# print(out)
# print(out["answer"])


In [ ]:
print("\nCitations:", out["citations"][:5])

In [ ]:
print("\nAnswer:", out["answer"])

In [ ]:
# print("\nHits:", out["hits"])
for x in out["hits"]:
    print(x)